# Criação da função de pré-processamento de dados, carregamento e geração de um novo arquivo .csv

In [ ]:
# Pré-passo. Carregar o df da última célula
df_clean = pd.read_csv(f"{NAME_FILE_2}.csv", encoding = "utf-8")

# 1. Carregar modelo spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Modelo spaCy não encontrado. Execute: python -m spacy download en_core_web_sm")
    raise
    
# 2. Declaração do conjunto de STOPWORDS
STOPWORDS_SET1 = {"palavra1","palavra2"}

# 3. Função única de limpeza
def limpar_texto(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    doc = nlp(text.lower())

    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and token.lemma_.lower() not in STOPWORDS_SET1
        and not token.is_punct
        and not token.is_space
        and not token.like_num
        and len(token.lemma_) > 2
    ]

    return " ".join(tokens)

# 4. Aplicar sobre o df_clean na coluna de resumos
print("Iniciando limpeza textual sobre a coluna 'abstract'...")

clean_text = df_clean['abstract'].apply(limpar_texto)

# 5. Criar novo DataFrame
df_custom = pd.DataFrame({
    'id': df_clean['id'],
    'year': df_clean['year'],
    'clean_text': clean_text,
    'all_areas': df_clean['all_areas']
})

# 6. Descartar linhas que contém clean_text vazio
count_before = len(df_custom)
df_custom = df_custom[df_custom['clean_text'].str.strip() != '']
df_custom = df_custom.reset_index(drop=True)
count_final = len(df_custom)

print(f"Total de artigos antes do descarte por texto vazio: {count_before}")
print(f"Total de artigos com clean_text válido: {count_final}")
print(f"Artigos descartados (clean_text vazio): {count_before - count_final}")

# 7. Exportar dataset final
NAME_FILE_3 = "insira-nome-do-arquivo-csv"
df_custom.to_csv(f"{NAME_FILE_3}.csv", index=False, encoding='utf-8')

print(f"Pré-processamento concluído! O arquivo '{NAME_FILE_3}.csv' com {len(df_custom)} artigos foi gerado com sucesso.")
